# Logistische regressie

Een simpele logistische regressie, met dezelfde folds en maten als `01_baseline.ipynb` en `03_decision_tree.ipynb`.

**LET OP: dit is DUMMY data.** De scores zeggen niets over echte modellen.

## 1. Data inladen

De testfamilies blijven dicht.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# De hoofdmap van de repo zoeken (daar staat pyproject.toml) en in sys.path zetten.
HOOFDMAP = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(HOOFDMAP))
DATA_MAP = HOOFDMAP / "outputs" / "candidate"

from src.candidate.data.features import selecteer_features
from src.candidate.data.labelling_rule import laad_benchmark, maak_configuratietabel, voldoende_categorieen

scenarios = pd.read_csv(DATA_MAP / "scenarios.csv")
configuraties = maak_configuratietabel(laad_benchmark(DATA_MAP / "dummy_benchmark.csv"))

with open(DATA_MAP / "split_manifest.json", encoding="utf-8") as bestand:
    manifest = json.load(bestand)

train = scenarios[~scenarios["family_id"].isin(manifest["test_families"])]
len(train)

1672

## 2. Features klaarmaken

Alleen gebruikersinvoer (`features.py`). Een leeg budget wordt 0, `has_energy_budget` laat zien dat er geen budget was. `hardware` wordt one-hot.

In [2]:
def maak_x_y(df):
    x, y = selecteer_features(df)
    x["max_energy_kwh_per_month"] = x["max_energy_kwh_per_month"].fillna(0)
    x = pd.get_dummies(x, columns=["hardware"], dtype=float)
    x["has_energy_budget"] = x["has_energy_budget"].astype(float)
    return x, y


x_train, y_train = maak_x_y(train)
x_train.head()

,input_length_tokens,min_macro_f1,max_latency_ms_per_doc,monthly_volume,max_energy_kwh_per_month,has_energy_budget,hardware_rtx3080,hardware_ryzen5_3600x
0,384,0.842,415.30,123624,0.911700,1.0,1.0,0.0
1,384,0.875,4.43,22447,0.129600,1.0,0.0,1.0
3,384,0.860,222.38,11577,0.000000,0.0,0.0,1.0
4,384,0.818,36.37,1353,0.002304,1.0,0.0,1.0
5,384,0.892,15.15,12923,0.001855,1.0,0.0,1.0


## 3. Trainen per fold

Elke fold is één keer de validatieset. De schaling zit in de pipeline, dus die wordt alleen op de trainfolds geleerd. Logistische regressie heeft schaling nodig omdat de features heel verschillende bereiken hebben.

In [3]:
def maak_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


delen = []
for fold in manifest["cv_folds"]:
    in_fold = train["family_id"].isin(fold["families"])

    model = maak_model()
    model.fit(x_train[~in_fold], y_train[~in_fold])

    validatie = train[in_fold].copy()
    validatie["voorspeld"] = model.predict(x_train[in_fold])
    delen.append(validatie)

resultaat = pd.concat(delen)
resultaat[["family_id", "label", "voorspeld"]].head()

,family_id,label,voorspeld
6,rtx3080__64,medium,medium
14,rtx3080__64,compact,compact
15,ryzen5_3600x__64,medium,geen
18,rtx3080__64,medium,medium
29,rtx3080__64,compact,compact


## 4. Confusion matrix

Rijen = echt label, kolommen = voorspelling.

In [4]:
labels = ["compact", "medium", "geen"]

matrix = pd.DataFrame(
    confusion_matrix(resultaat["label"], resultaat["voorspeld"], labels=labels),
    index=labels,
    columns=labels,
)
matrix

,compact,medium,geen
compact,434,14,73
medium,11,375,128
geen,85,92,460


## 5. Scores

Dezelfde maten als bij de baseline.

In [5]:
def scores(df):
    gekozen = df[df["voorspeld"] != "geen"]
    violations = sum(
        rij["voorspeld"] not in voldoende_categorieen(configuraties, rij.to_dict())
        for _, rij in gekozen.iterrows()
    )

    kansen = df[df["label"] != "geen"]
    gemiste_kansen = int((kansen["voorspeld"] == "geen").sum())

    return {
        "macro_f1": f1_score(df["label"], df["voorspeld"], labels=labels, average="macro"),
        "violations": violations,
        "keuzes": len(gekozen),
        "violation_rate": violations / len(gekozen),
        "gemiste_kansen": gemiste_kansen,
        "kansen": len(kansen),
        "gemiste_kans_rate": gemiste_kansen / len(kansen),
    }


scores(resultaat)

{'macro_f1': 0.7628105670263757,
 'violations': 188,
 'keuzes': 1011,
 'violation_rate': 0.18595450049455983,
 'gemiste_kansen': 201,
 'kansen': 1035,
 'gemiste_kans_rate': 0.19420289855072465}

## 6. Scores per inputlengte

In [6]:
pd.DataFrame(
    {lengte: scores(deel) for lengte, deel in resultaat.groupby("input_length_tokens")}
).T.round(3)

,macro_f1,violations,keuzes,violation_rate,gemiste_kansen,kansen,gemiste_kans_rate
64,0.740,28.0,263.0,0.106,44.0,279.0,0.158
128,0.748,49.0,226.0,0.217,30.0,213.0,0.141
192,0.732,55.0,239.0,0.230,33.0,222.0,0.149
384,0.781,30.0,164.0,0.183,40.0,174.0,0.230
512,0.716,26.0,119.0,0.218,54.0,147.0,0.367


## 7. De gewichten bekijken

Eén model op alle trainfamilies. Per klasse: een positief gewicht maakt die klasse waarschijnlijker. De features zijn geschaald, dus de gewichten zijn onderling te vergelijken.

In [8]:
model = maak_model()
model.fit(x_train, y_train)

regressie = model.named_steps["logisticregression"]
pd.DataFrame(regressie.coef_, index=regressie.classes_, columns=x_train.columns).T.round(2)

,compact,geen,medium
input_length_tokens,-0.33,0.55,-0.22
min_macro_f1,-2.11,0.92,1.19
max_latency_ms_per_doc,0.51,-0.88,0.37
monthly_volume,-0.04,0.13,-0.09
max_energy_kwh_per_month,0.05,-0.16,0.11
has_energy_budget,-0.18,0.40,-0.22
hardware_rtx3080,0.28,-0.44,0.16
hardware_ryzen5_3600x,-0.28,0.44,-0.16
